# FRQ GT Generation Demo: Spatial Perception + Spatial Understanding

This notebook demonstrates a conservative FRQ workflow on one nuScenes 5-frame 360 stitched context:

1. Load one object-level group from `nuscenes_test/formatted_scenes`.
2. Build two FRQ examples: spatial perception (`SP-7`) and spatial understanding (`SU-7`).
3. Use existing MCQ ground-truth facts to create rule-generated reasoning steps and a rule-generated answer.
4. Ask Qwen to polish the rule-generated answer without changing facts.
5. Ask Qwen to answer the same FRQ from the 5 images, then compare model response vs generated GT.

The key idea is: facts come from programmatic GT; Qwen only writes the language.


In [ ]:
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

REPO = Path('/home/rgao727/Spatial_Temporal_Intelligence')
FORMATTED_SCENES = REPO / 'nuscenes_test' / 'formatted_scenes'
TASKS_JSON = REPO / 'nuscenes_test' / 'questions_with_answers_all.json'

# Pick one object-level group that already has MCQ GT.
SCENE_ID = 'scene_001'
GROUP_ID = 'group_001_obj01'
SOURCE_GROUP_FILE = f'{SCENE_ID}/{GROUP_ID}_vehicle_annotations.json'

print('formatted scenes:', FORMATTED_SCENES)
print('tasks json:', TASKS_JSON)
print('source group:', SOURCE_GROUP_FILE)

In [ ]:
def load_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def flatten_generated_tasks(tasks_json):
    out = []
    for task_id, payload in tasks_json.get('generated_answers', {}).items():
        for task in payload.get('tasks', []):
            row = dict(task)
            row['question_id'] = task_id
            out.append(row)
    return out


def resolve_5frame_grid_paths(group_payload, formatted_scenes_dir: Path):
    scene_id = group_payload['scene_id']
    frame_indices = group_payload['frame_indices_1based']
    scene_dir = formatted_scenes_dir / scene_id
    image_paths = []
    for idx in frame_indices:
        matches = sorted(scene_dir.glob(f'{idx:03d}_*_grid.jpg'))
        if not matches:
            raise FileNotFoundError(f'Missing grid image for frame {idx} in {scene_dir}')
        image_paths.append(matches[0])
    return image_paths


tasks_json = load_json(TASKS_JSON)
all_tasks = flatten_generated_tasks(tasks_json)
group_payload = load_json(FORMATTED_SCENES / SOURCE_GROUP_FILE)
image_paths = resolve_5frame_grid_paths(group_payload, FORMATTED_SCENES)

group_tasks = [
    t for t in all_tasks
    if t.get('scene_id') == SCENE_ID and t.get('group_id') == GROUP_ID
]
print('num group tasks:', len(group_tasks))
print('image paths:')
for p in image_paths:
    print(' ', p)

In [ ]:
# Visualize the 5 consecutive 360-degree stitched frames.
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
for i, (ax, path) in enumerate(zip(axes, image_paths), start=1):
    img = Image.open(path).convert('RGB')
    ax.imshow(img)
    ax.set_title(f'frame {i}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
def option_text(task):
    gt = str(task.get('ground_truth', '')).strip()
    choices = task.get('choices', {})
    return str(choices.get(gt, gt))


def collect_mcq_facts(source_ids):
    rows = []
    wanted = set(source_ids)
    for t in group_tasks:
        qid = t.get('question_id') or t.get('id')
        if qid in wanted and t.get('question_format') == 'MCQ':
            rows.append({
                'id': qid,
                'question': t.get('question', ''),
                'ground_truth_key': t.get('ground_truth', ''),
                'ground_truth_text': option_text(t),
                'choices': t.get('choices', {}),
                'object_reference': t.get('object_reference', ''),
            })
    order = {qid: i for i, qid in enumerate(source_ids)}
    return sorted(rows, key=lambda x: order.get(x['id'], 999))


FRQ_CONFIGS = {
    'SP': {
        'question_id': 'SP-7',
        'source_ids': ['SP-1', 'SP-2', 'SP-3', 'SP-4', 'SP-5', 'SP-6'],
        'question': (
            'Describe the spatial relationship between the ego vehicle and <obj>, including its relative position, '
            'orientation of motion, and approximate distance.'
        ),
    },
    'SU': {
        'question_id': 'SU-7',
        'source_ids': ['SU-1', 'SU-2', 'SU-3', 'SU-4', 'SU-5', 'SU-6'],
        'question': (
            'Describe the overall spatial situation of the ego vehicle in the scene, including existing physical '
            'constraints, available drivable space, risk level, and feasible maneuvers.'
        ),
    },
}

examples = {}
for mode, cfg in FRQ_CONFIGS.items():
    facts = collect_mcq_facts(cfg['source_ids'])
    object_ref = next((r['object_reference'] for r in facts if r.get('object_reference')), 'the selected object')
    question = cfg['question'].replace('<obj>', object_ref)
    examples[mode] = {'cfg': cfg, 'facts': facts, 'object_reference': object_ref, 'question': question}

for mode, ex in examples.items():
    print('=' * 80)
    print('FRQ example:', mode)
    print('Question:', ex['question'])
    print('Object:', ex['object_reference'])
    print('Structured MCQ facts:')
    for row in ex['facts']:
        print(f"  {row['id']}: {row['ground_truth_key']} -> {row['ground_truth_text']}")


In [ ]:
def fact_map(facts):
    return {f['id']: f['ground_truth_text'] for f in facts}


def get_fact(facts_by_id, qid, default='unknown'):
    return str(facts_by_id.get(qid, default))


def build_rule_generated_frq(mode, facts, object_reference):
    f = fact_map(facts)
    if mode == 'SP':
        motion = get_fact(f, 'SP-1', 'unknown motion')
        position = get_fact(f, 'SP-2', 'unknown position')
        distance = get_fact(f, 'SP-3', 'unknown distance')
        lane = get_fact(f, 'SP-4', 'unknown lane relation')
        visibility = get_fact(f, 'SP-5', 'unknown visibility')
        ground = get_fact(f, 'SP-6', 'unknown ground-plane state')
        reasoning_steps = [
            f'The target is {object_reference}, so the answer should describe that object relative to ego.',
            f'The relative position is {position}, which anchors the spatial relationship.',
            f'The distance bin is {distance}, so the answer should state approximate range without inventing a precise meter value.',
            f'The lane relation is {lane}, which explains where the object sits with respect to ego lanes.',
            f'The motion label is {motion}, while visibility is {visibility} and ground-plane state is {ground}.',
        ]
        rule_answer = (
            f'The selected object is {object_reference}. It is located {position.lower()} of the ego vehicle at an approximate distance of {distance.lower()}. '
            f'Its lane relation is {lane.lower()}, and its motion is described as {motion.lower()}. '
            f'The object is {visibility.lower()} and appears to be {ground.lower()}.'
        )
    elif mode == 'SU':
        constraint = get_fact(f, 'SU-1', 'unknown constraint')
        drivable = get_fact(f, 'SU-2', 'unknown drivable region')
        risk = get_fact(f, 'SU-3', 'unknown risk')
        lane_change = get_fact(f, 'SU-4', 'unknown lane-change clearance')
        maneuver = get_fact(f, 'SU-5', 'unknown maneuver')
        density = get_fact(f, 'SU-6', 'unknown density')
        reasoning_steps = [
            f'The selected object is {object_reference}, so the answer should focus on how that object affects ego driving.',
            f'The physical constraint label is {constraint}, which determines whether the object blocks or interferes with the ego path.',
            f'The drivable-space label is {drivable}, and the lane-change label is {lane_change}; together they describe available maneuvering space.',
            f'The risk label is {risk}, so the recommended action should be consistent with that risk level rather than overreacting.',
            f'The feasible maneuver is {maneuver}, while the surrounding density is {density}.',
        ]
        rule_answer = (
            f'The selected object is {object_reference}. It imposes {constraint.lower()}, and the safely drivable region is {drivable.lower()}. '
            f'The risk level is {risk.lower()}, with lane-change clearance described as {lane_change.lower()}. '
            f'The feasible maneuver is {maneuver.lower()} in a scene with {density.lower()} surrounding traffic density.'
        )
    else:
        raise ValueError(mode)
    return {'reasoning_steps': reasoning_steps, 'rule_answer': rule_answer}


for mode, ex in examples.items():
    rule_generated = build_rule_generated_frq(mode, ex['facts'], ex['object_reference'])
    ex.update(rule_generated)
    print('=' * 80)
    print(mode, 'rule-generated reasoning steps:')
    for i, step in enumerate(ex['reasoning_steps'], start=1):
        print(f'{i}. {step}')
    print('\nRule-generated answer:')
    print(ex['rule_answer'])


In [ ]:
# Qwen setup. Leave HF_TOKEN empty if the model is already cached and accessible.
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_NAME = 'Qwen/Qwen3-VL-30B-A3B-Instruct'
HF_TOKEN = os.environ.get('HF_TOKEN', '')
CACHE_DIR = os.environ.get('HF_HOME', '/data2/rgao727/hf_cache_store')

min_pixels = 256 * 28 * 28
max_pixels = 768 * 28 * 28

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=HF_TOKEN or None,
    cache_dir=CACHE_DIR,
    min_pixels=min_pixels,
    max_pixels=max_pixels,
)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN or None,
    cache_dir=CACHE_DIR,
    trust_remote_code=True,
)
model.eval()
print('Loaded', MODEL_NAME)

In [ ]:
def apply_chat(processor, messages, images=None):
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if images is None:
        return processor(text=[text], return_tensors='pt', padding=True)
    return processor(text=[text], images=images, return_tensors='pt', padding=True)


def generate_text(messages, images=None, max_new_tokens=384):
    inputs = apply_chat(processor, messages, images=images)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    prompt_len = inputs['input_ids'].shape[1]
    gen_ids = output_ids[:, prompt_len:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()


def facts_as_text(facts):
    lines = []
    for f in facts:
        lines.append(f"- {f['id']}: {f['question']}")
        lines.append(f"  Ground truth: {f['ground_truth_text']}")
    return '\n'.join(lines)

In [ ]:
# Step 1: use Qwen as a language polisher for GT, grounded only in rule-generated facts and steps.
def polish_gt_with_qwen(mode, ex):
    gt_polish_prompt = f"""
You are writing a ground-truth free-response answer for an autonomous-driving QA benchmark.

Use only the provided structured facts, rule-generated reasoning steps, and rule-generated answer. Do not add new objects, distances, lane relations, weather, traffic light states, or actions. Do not mention option letters. Preserve every safety-critical fact.

Question:
{ex['question']}

Target object:
{ex['object_reference']}

Structured MCQ facts:
{facts_as_text(ex['facts'])}

Rule-generated reasoning steps:
{chr(10).join(f'- {s}' for s in ex['reasoning_steps'])}

Rule-generated answer:
{ex['rule_answer']}

Polish the rule-generated answer into 2-4 concise, natural sentences. Keep the causal reasoning explicit but do not expose chain-of-thought or numbered steps.
""".strip()

    gt_messages = [{'role': 'user', 'content': [{'type': 'text', 'text': gt_polish_prompt}]}]
    return generate_text(gt_messages, images=None, max_new_tokens=256)


for mode, ex in examples.items():
    print('=' * 80)
    print('Polishing GT for', mode)
    ex['qwen_polished_gt'] = polish_gt_with_qwen(mode, ex)
    print(ex['qwen_polished_gt'])


In [ ]:
# Step 2: ask Qwen to answer each FRQ from the 5 visual frames only.
def answer_from_images_with_qwen(ex):
    images = [Image.open(p).convert('RGB') for p in image_paths]
    visual_prompt = f"""
You are given 5 consecutive 360-degree stitched driving frames from nuScenes. The first 4 frames provide temporal context and the 5th frame is the query frame.

Question:
{ex['question']}

Answer in 2-4 concise sentences. Focus only on what is relevant to the question.
""".strip()

    visual_messages = [{
        'role': 'user',
        'content': [{'type': 'image'} for _ in images] + [{'type': 'text', 'text': visual_prompt}],
    }]
    try:
        return generate_text(visual_messages, images=images, max_new_tokens=384)
    finally:
        for im in images:
            im.close()


for mode, ex in examples.items():
    print('=' * 80)
    print('Visual answer for', mode)
    ex['qwen_visual_answer'] = answer_from_images_with_qwen(ex)
    print(ex['qwen_visual_answer'])


In [ ]:
for mode, ex in examples.items():
    print('=' * 100)
    print('MODE')
    print(mode)
    print('\nQUESTION')
    print(ex['question'])
    print('\nSTRUCTURED FACTS')
    print(facts_as_text(ex['facts']))
    print('\nRULE-GENERATED REASONING STEPS')
    for i, step in enumerate(ex['reasoning_steps'], start=1):
        print(f'{i}. {step}')
    print('\nRULE-GENERATED ANSWER')
    print(ex['rule_answer'])
    print('\nQWEN-POLISHED GT')
    print(ex.get('qwen_polished_gt', ''))
    print('\nQWEN VISUAL ANSWER')
    print(ex.get('qwen_visual_answer', ''))
